# 1. Setup
**Approach 1: Supervised contrastive learning.** Multi-task CNN plus two supervised contrastive losses.
Images with the same disease are pulled together in a disease space, whatever the crop.
Images with the same crop are pulled together in a crop space, whatever the disease.
Idea: disease features that ignore the crop should transfer better to unseen crop-disease pairs.

In [1]:
import copy
import sys
from pathlib import Path

sys.path.append( str( Path.cwd().parent ) )  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from common import config
from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "approach1_contrastive"
DEBUG = False  # Full training and evaluation data, one epoch.
HP = {
	"backbone": "resnet50", 
	"lr": 1e-4, 
	"weight_decay": 1e-4, 
	"epochs": 30, 
	"batch_size": 128, 
	"optimizer": "AdamW",
	"label_smoothing": 0.1, 
	"projection_dim": 128, 
	"temperature": 0.1, 
	"contrastive_weight": 0.1
}
# batch_size is larger than the baselines so more images share a label in each batch
EPOCHS = HP["epochs"]
RESULTS_DIR = config.RESULTS_DIR / "approach1"
config.RESULTS_DIR = RESULTS_DIR
RESULTS_DIR.mkdir( parents = True, exist_ok = True )
set_seed( 42 )

In [2]:
# For debugging, use a smaller batch size and no workers to avoid issues with multiprocessing on Windows.
# Remember remove this when doing full training.
'''
DEBUG = True
HP["batch_size"] = 16
HP["epochs"] = 1
config.NUM_WORKERS = 0
'''

'\nDEBUG = True\nHP["batch_size"] = 16\nHP["epochs"] = 1\nconfig.NUM_WORKERS = 0\n'

# 2. Data

In [3]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
	HP["batch_size"], norm = "imagenet", debug = DEBUG
)
pairs = get_pairs().to( DEVICE )

# 3. Model

In [4]:
def projector(in_features):
	return nn.Sequential(
		nn.Linear( in_features, in_features ), nn.ReLU(), nn.Linear( in_features, HP["projection_dim"] )
	)


class ContrastiveNet( nn.Module ):
	"""Shared backbone, two classifier heads and two projection heads (training only)."""

	def __init__(self, backbone_name, pretrained = True):
		super().__init__()
		self.backbone = timm.create_model( backbone_name, pretrained = pretrained, num_classes = 0 )
		width = self.backbone.num_features
		self.crop_head, self.disease_head = nn.Linear( width, NUM_CROPS ), nn.Linear( width, NUM_DISEASES )
		self.crop_projector, self.disease_projector = projector( width ), projector( width )

	def forward(self, images):
		features = self.backbone( images )
		logits = (self.crop_head( features ), self.disease_head( features ))
		projections = (self.crop_projector( features ), self.disease_projector( features ))
		return logits, projections


def supcon_loss(projections, labels):
	"""Return supervised contrastive loss and the fraction of anchors with a positive."""
	features = F.normalize( projections.float(), dim = 1 )
	self_mask = torch.eye( len( labels ), dtype = torch.bool, device = labels.device )
	similarity = (features @ features.T / HP["temperature"]).masked_fill( self_mask, -1e9 )
	positives = (labels[:, None] == labels[None, :]) & ~self_mask
	has_positive = positives.any( dim = 1 )  # skip anchors with no partner in this batch
	anchor_fraction = has_positive.float().mean()
	if not has_positive.any():
		return similarity.sum() * 0, anchor_fraction
	log_prob = similarity.log_softmax( dim = 1 )
	mean_positive = (log_prob * positives).sum( dim = 1 )[has_positive] / positives.sum( dim = 1 )[has_positive]
	return -mean_positive.mean(), anchor_fraction


model = ContrastiveNet( HP["backbone"] ).to( DEVICE )


@torch.no_grad()
def predict(images):
	model.eval()
	(crop_logits, disease_logits), _ = model( images.to( DEVICE ) )
	crop_log_prob = F.log_softmax( crop_logits.float(), dim = 1 )
	disease_log_prob = F.log_softmax( disease_logits.float(), dim = 1 )
	return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [5]:
optimizer = torch.optim.AdamW( model.parameters(), lr = HP["lr"], weight_decay = HP["weight_decay"] )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR( optimizer, T_max = EPOCHS )
criterion = nn.CrossEntropyLoss( label_smoothing = HP["label_smoothing"] )
best_val, best_state = -1.0, None

for epoch in range( EPOCHS ):
	model.train()
	epoch_ce = epoch_contrastive = epoch_total = 0.0
	epoch_crop_anchor_fraction = epoch_disease_anchor_fraction = 0.0
	num_batches = 0
	for images, crop_labels, disease_labels, _ in tqdm(
		train_loader, desc = f"Epoch {epoch + 1}/{EPOCHS}", unit = "batch", dynamic_ncols = True
	):
		images = images.to( DEVICE )
		crop_labels, disease_labels = crop_labels.to( DEVICE ), disease_labels.to( DEVICE )
		with torch.autocast( DEVICE.type, dtype = torch.bfloat16, enabled = DEVICE.type == "cuda" ):
			(crop_logits, disease_logits), (crop_proj, disease_proj) = model( images )
			ce_loss = criterion( crop_logits, crop_labels ) + criterion( disease_logits, disease_labels )
		crop_contrastive, crop_anchor_fraction = supcon_loss( crop_proj, crop_labels )
		disease_contrastive, disease_anchor_fraction = supcon_loss( disease_proj, disease_labels )
		contrastive = crop_contrastive + disease_contrastive
		loss = ce_loss + HP["contrastive_weight"] * contrastive
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		epoch_ce += ce_loss.detach()
		epoch_contrastive += contrastive.detach()
		epoch_total += loss.detach()
		epoch_crop_anchor_fraction += crop_anchor_fraction.detach()
		epoch_disease_anchor_fraction += disease_anchor_fraction.detach()
		num_batches += 1
	scheduler.step()
	val_top1 = val_accuracy( predict, val_loader )
	print(
		f"epoch {epoch + 1}: mean CE {epoch_ce.item() / num_batches:.3f}, "
		f"mean contrastive {epoch_contrastive.item() / num_batches:.3f}, "
		f"mean total {epoch_total.item() / num_batches:.3f}, "
		f"crop anchors with positives {100 * epoch_crop_anchor_fraction.item() / num_batches:.1f}%, "
		f"disease anchors with positives {100 * epoch_disease_anchor_fraction.item() / num_batches:.1f}%, "
		f"val top-1 {val_top1:.2f}%"
	)
	if val_top1 > best_val:  # keep the best epoch by val, never by test
		best_val, best_state = val_top1, copy.deepcopy( model.state_dict() )

model.load_state_dict( best_state )

Epoch 1/30:   0%|          | 0/57 [00:11<?, ?batch/s]

epoch 1: mean CE 8.436, mean contrastive 7.927, mean total 9.228, crop anchors with positives 91.1%, disease anchors with positives 78.5%, val top-1 10.55%


Epoch 2/30:   0%|          | 0/57 [00:00<?, ?batch/s]

epoch 2: mean CE 6.932, mean contrastive 6.939, mean total 7.626, crop anchors with positives 91.0%, disease anchors with positives 79.0%, val top-1 14.67%


Epoch 3/30:   0%|          | 0/57 [00:00<?, ?batch/s]

...

Epoch 29/30:   0%|          | 0/57 [00:00<?, ?batch/s]

epoch 29: mean CE 2.384, mean contrastive 4.162, mean total 2.801, crop anchors with positives 91.7%, disease anchors with positives 78.6%, val top-1 73.62%


Epoch 30/30:   0%|          | 0/57 [00:00<?, ?batch/s]

epoch 30: mean CE 2.382, mean contrastive 4.154, mean total 2.798, crop anchors with positives 90.5%, disease anchors with positives 79.0%, val top-1 73.10%


<All keys matched successfully>

# 5. Evaluation

In [6]:
results = evaluate( predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust = True )

{
  "model": "approach1_contrastive",
  "seen_top1": 56.6,
  "seen_top5": 81.0,
  "unseen_top1": 2.9,
  "unseen_top5": 26.04,
  "harmonic_mean": 5.52,
  "crop_acc": 83.45,
  "disease_acc": 60.1,
  "robustness": {
    "brightness": {
      "top1": 54.1,
      "drop": 2.5
    },
    "blur": {
      "top1": 35.8,
      "drop": 20.8
    },
    "noise": {
      "top1": 24.55,
      "drop": 32.05
    }
  }
}


# 6. Save

In [7]:
CHECKPOINT_DIR.mkdir( parents = True, exist_ok = True )
torch.save(
	{ "model_name": MODEL_NAME, "state_dict": model.state_dict(), "hp": HP }, CHECKPOINT_DIR / f"{MODEL_NAME}.pt"
)